# Superstore Sales Analytics: ETL Pipeline
**Author :** Thejas K S  
**Role :** Data Science & Analytics Intern (Future Interns)

## Project Overview
This notebook executes the Extract, Transform, and Load (ETL) pipeline for the Superstore retail dataset. It cleans the raw data, engineers business-critical features (like Profit Margin and Delivery Speed), and exports a production-ready dataset for the Power BI dashboard.

In [1]:
#Importing Libraries
import pandas as pd
import numpy as np
import os

print("Libraries successfully loaded.")

Libraries successfully loaded.


## 1. Extract
We load the raw Kaggle dataset. The original file uses a legacy `windows-1252` encoding, which we must specify to avoid UTF-8 decoding errors.

In [2]:
# Load the raw data
raw_file_path = '../data/raw/Superstore Dataset/Sample - Superstore.csv'
df = pd.read_csv(raw_file_path, encoding='windows-1252')

print(f"Data Extracted: {df.shape[0]} rows and {df.shape[1]} columns.")

Data Extracted: 9994 rows and 21 columns.


## 1.5 Data Auditing (Exploratory Data Analysis)
Before transforming the data, we must audit the raw file to identify missing values, incorrect data types, and logical anomalies (The "Sneaky Mess").

In [3]:
# 1. The Basic Audit: Check structure and data types
print("--- DATA STRUCTURE ---")
print(df.info())

# 2. The Quality Audit: Check for missing values
print("\n--- MISSING VALUES ---")
print(df.isnull().sum())

# 3. The Statistical Audit: Check for logical impossibilities (e.g., negative sales)
print("\n--- NUMERIC STATISTICS ---")
display(df.describe())

--- DATA STRUCTURE ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null 

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55190.379428,229.858001,3.789574,0.156203,28.656896
std,2885.163629,32063.693350,623.245101,2.225110,0.206452,234.260108
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000


## 2. Transform (Data Cleaning & Feature Engineering)
In this phase, we address structural issues and engineer new KPIs for the dashboard.
* **Standardization:** Convert column headers to snake_case.
* **Cleaning:** Drop the useless `row_id` index and remove any duplicate transactions.
* **Type Conversion:** Fix text-based dates into Pandas datetime objects. Fix the `postal_code` integer bug by padding it back to a 5-digit string.
* **Feature Engineering:** Calculate `days_to_ship` (logistics KPI) and `profit_margin` (financial KPI).

In [5]:
# A. Standardize Headers
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')

# B. Drop useless columns and duplicate rows
if 'row_id' in df.columns:
    df.drop(columns=['row_id'], inplace=True)
df.drop_duplicates(inplace=True)

# C. Fix Data Types
# Convert text dates to datetime objects (handling mixed slash/dash formats)
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed')
df['ship_date'] = pd.to_datetime(df['ship_date'], format='mixed')

# Fix the Postal Code bug (Pad with leading zeros so '1040' becomes '01040')
df['postal_code'] = df['postal_code'].astype(str).str.split('.').str[0].str.zfill(5)

# D. Feature Engineering
# Extract date components for easier filtering in Power BI
df['order_year'] = df['order_date'].dt.year
df['order_month_num'] = df['order_date'].dt.month
df['order_month_name'] = df['order_date'].dt.month_name()

# Calculate delivery speed
df['days_to_ship'] = (df['ship_date'] - df['order_date']).dt.days

# Calculate Profit Margin (Handle potential division by zero)
df['profit_margin'] = np.where(df['sales'] > 0, (df['profit'] / df['sales']).round(4), 0)

print("Transformation complete. Sample of engineered features:")
display(df[['order_date', 'days_to_ship', 'sales', 'profit', 'profit_margin']].head())

Transformation complete. Sample of engineered features:


,order_date,days_to_ship,sales,profit,profit_margin
0,2016-11-08,3,261.9600,41.9136,0.1600
1,2016-11-08,3,731.9400,219.5820,0.3000
2,2016-06-12,4,14.6200,6.8714,0.4700
3,2015-10-11,7,957.5775,-383.0310,-0.4000
4,2015-10-11,7,22.3680,2.5164,0.1125


## 3. Load
The cleaned and enriched dataframe is now exported to the `processed` directory. This file will serve as the single source of truth for the Power BI dashboard.

In [6]:
# Define destination
output_dir = '../data/processed'
output_file = f'{output_dir}/superstore_sales_clean.csv'

# Ensure the directory exists
os.makedirs(output_dir, exist_ok=True)

# Save the clean data without the index
df.to_csv(output_file, index=False)

print(f"ETL Pipeline Complete! Clean data saved to: {output_file}")
print(f"Final Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

ETL Pipeline Complete! Clean data saved to: ../data/processed/superstore_sales_clean.csv
Final Dataset Shape: 9993 rows, 25 columns.


## Load (Save Processed Data)
Exporting the final, clean dataset to the `processed` folder for Power BI ingestion.

In [7]:
# Load the processed data
processed_file_path = '../data/processed/superstore_sales_clean.csv'
df = pd.read_csv(processed_file_path)

print(f"Data Extracted: {df.shape[0]} rows and {df.shape[1]} columns.")

Data Extracted: 9993 rows and 25 columns.


In [8]:
# 1. The Basic Audit: Check structure and data types
print("--- DATA STRUCTURE ---")
print(df.info())

# 2. The Quality Audit: Check for missing values
print("\n--- MISSING VALUES ---")
print(df.isnull().sum())

# 3. The Statistical Audit: Check for logical impossibilities (e.g., negative sales)
print("\n--- NUMERIC STATISTICS ---")
display(df.describe())

--- DATA STRUCTURE ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9993 entries, 0 to 9992
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          9993 non-null   object 
 1   order_date        9993 non-null   object 
 2   ship_date         9993 non-null   object 
 3   ship_mode         9993 non-null   object 
 4   customer_id       9993 non-null   object 
 5   customer_name     9993 non-null   object 
 6   segment           9993 non-null   object 
 7   country           9993 non-null   object 
 8   city              9993 non-null   object 
 9   state             9993 non-null   object 
 10  postal_code       9993 non-null   int64  
 11  region            9993 non-null   object 
 12  product_id        9993 non-null   object 
 13  category          9993 non-null   object 
 14  sub_category      9993 non-null   object 
 15  product_name      9993 non-null   object 
 16  sales             9

,postal_code,sales,quantity,discount,profit,order_year,order_month_num,days_to_ship,profit_margin
count,9993.000000,9993.000000,9993.000000,9993.000000,9993.000000,9993.000000,9993.000000,9993.000000,9993.000000
mean,55191.576403,229.852846,3.789753,0.156188,28.660971,2015.722406,7.810067,3.958171,0.120330
std,32065.074478,623.276074,2.225149,0.206457,234.271476,1.123479,3.284598,1.747654,0.466775
min,1040.000000,0.444000,1.000000,0.000000,-6599.978000,2014.000000,1.000000,0.000000,-2.750000
25%,23223.000000,17.280000,2.000000,0.000000,1.731000,2015.000000,5.000000,3.000000,0.075000
50%,56560.000000,54.480000,3.000000,0.200000,8.671000,2016.000000,9.000000,4.000000,0.270000
75%,90008.000000,209.940000,5.000000,0.200000,29.364000,2017.000000,11.000000,5.000000,0.362500
max,99301.000000,22638.480000,14.000000,0.800000,8399.976000,2017.000000,12.000000,7.000000,0.500000


## 4. Verification (Raw vs. Processed Comparison)
To confirm the ETL pipeline successfully cleaned the dataset, we compare the initial raw data state against our new processed data state.

In [10]:
print("=========================================")
print("       ETL VERIFICATION REPORT           ")
print("=========================================")

# 1. Load the processed file the CORRECT way so Pandas doesn't get amnesia
verification_df = pd.read_csv(
    '../data/processed/superstore_sales_clean.csv', 
    parse_dates=['order_date', 'ship_date'], 
    dtype={'postal_code': str} # Forces postal_code to stay as a string!
)

# 2. Row Count Check (Did we drop the duplicates?)
raw_rows = 9994  
processed_rows = len(verification_df)
print(f"1. Duplicate Check:")
print(f"   Raw Rows:       {raw_rows}")
print(f"   Processed Rows: {processed_rows}")
print(f"   Status:         {raw_rows - processed_rows} duplicates removed.\n")

# 3. Date Data Type Check (Did we fix the dates?)
print(f"2. Data Type Check (order_date):")
print(f"   Raw Type:       object (Text)")
print(f"   Processed Type: {verification_df['order_date'].dtype}")
print(f"   Status:         Fixed to Datetime.\n")

# 4. Postal Code Bug Check (Did we fix the lost zeros?)
print(f"3. Postal Code Bug Fix (Checking minimum length):")
min_postal_len = verification_df['postal_code'].str.len().min()
print(f"   Processed min length: {min_postal_len} characters")
if min_postal_len == 5:
    print("   Status:         SUCCESS. All leading zeros restored.\n")
else:
    print("   Status:         FAILED. Some codes are too short.\n")

# 5. Feature Engineering Check (Did we add the new business columns?)
print(f"4. Engineered Business Metrics Added:")
print(f"   - days_to_ship:  {verification_df['days_to_ship'].dtype}")
print(f"   - profit_margin: {verification_df['profit_margin'].dtype}")
print(f"   - order_year:    {verification_df['order_year'].dtype}")
print("=========================================")

       ETL VERIFICATION REPORT           
1. Duplicate Check:
   Raw Rows:       9994
   Processed Rows: 9993
   Status:         1 duplicates removed.

2. Data Type Check (order_date):
   Raw Type:       object (Text)
   Processed Type: datetime64[ns]
   Status:         Fixed to Datetime.

3. Postal Code Bug Fix (Checking minimum length):
   Processed min length: 5 characters
   Status:         SUCCESS. All leading zeros restored.

4. Engineered Business Metrics Added:
   - days_to_ship:  int64
   - profit_margin: float64
   - order_year:    int64
